# Module 10 - Programming Assignment

## Directions

1. Change the name of this file to be your JHED id as in `jsmith299.ipynb`. Because sure you use your JHED ID (it's made out of your name and not your student id which is just letters and numbers).
2. Make sure the notebook you submit is cleanly and fully executed. I do not grade unexecuted notebooks.
3. Submit your notebook back in Blackboard where you downloaded this file.

*Provide the output **exactly** as requested*

# Forward Planner

## Unify

Use the accompanying `unification.py` file for unification. For this assignment, you're almost certainly going to want to be able to:

1. specify the problem in terms of S-expressions.
2. parse them.
3. work with the parsed versions.

`parse` and `unification` work exactly like the programming assignment for last time.

In [86]:
from unification import parse, unification

## Forward Planner

In this assigment, you're going to implement a Forward Planner. What does that mean? If you look in your book, you will not find pseudocode for a forward planner. It just says "use state space search" but this is less than helpful and it's a bit more complicated than that. **(but please please do not try to implement STRIPS or GraphPlan...that is wrong).**

At a high level, a forward planner takes the current state of the world $S_0$ and attempts to derive a plan, basically by Depth First Search. We have all the ingredients we said we would need in Module 1: states, actions, a transition function and a goal test. We have a set of predicates that describe a state (and therefore all possible states), we have actions and we have, at least, an implicit transition function: applying an action in a state causes the state to change as described by the add and delete lists.

Let's say we have a drill that's an item, two places such as home and store, and we know that I'm at home and the drill is at the store and I want to go buy a drill (have it be at home). We might represent that as:

<code>
start_state = [
    "(item Saw)",
    "(item Drill)",
    "(place Home)",
    "(place Store)",
    "(place Bank)",
    "(agent Me)",
    "(at Me Home)",
    "(at Saw Store)",
    "(at Drill Store)",
    "(at Money Bank)"
]
</code>

And we have a goal state:

<code>
goal = [
    "(item Saw)",
    "(item Drill)",
    "(place Home)",
    "(place Store)",
    "(place Bank)",
    "(agent Me)",
    "(at Me Home)",
    "(at Drill Me)",
    "(at Saw Store)",
    "(at Money Bank)"
]
</code>

The actions/operators are:

<code>
actions = {
    "drive": {
        "action": "(drive ?agent ?from ?to)",
        "conditions": [
            "(agent ?agent)",
            "(place ?from)",
            "(place ?to)",
            "(at ?agent ?from)"
        ],
        "add": [
            "(at ?agent ?to)"
        ],
        "delete": [
            "(at ?agent ?from)"
        ]
    },
    "buy": {
        "action": "(buy ?purchaser ?seller ?item)",
        "conditions": [
            "(item ?item)",
            "(place ?seller)",
            "(agent ?purchaser)",
            "(at ?item ?seller)",
            "(at ?purchaser ?seller)"
        ],
        "add": [
            "(at ?item ?purchaser)"
        ],
        "delete": [
            "(at ?item ?seller)"
        ]
    }
}
</code>

These will all need to be parsed from s-expressions to the underlying Python representation before you can use them. You might as well do it at the start of your algorithm, once. The order of the conditions is *not* arbitrary. It is much, much better for the unification and backtracking if you have the "type" predicates (item, place, agent) before the more complex ones. Trust me on this.

As for the algorithm itself, there is going to be an *outer* level of search and an *inner* level of search.

The *outer* level of search that is exactly what I describe here: you have a state, you generate successor states by applying actions to the current state, you examine those successor states as we did at the first week of the semester and if one is the goal you stop, if you see a repeat state, you put it on the explored list (you should implement graph search not tree search). What could be simpler?

It turns out the Devil is in the details. There is an *inner* level of search hidden in "you generate successor states by applying actions to the current state". Where?

How do you know if an action applies in a state? Only if the preconditions successfully unify with the current state. That seems easy enough...you check each predicate in the conditions to see if it unifies with the current state and if it does, you use the substitution list on the action, the add and delete lists and create the successor state based on them.

Except for one small problem...there may be more than one way to unify an action with the current state. You must essentially search for all successful unifications of the candidate action and the current state. This is where my question through the semester appliesm, "how would you modify state space search to return all the paths to the goal?"

Unification can be seen as state space search by trying to unify the first precondition with the current state, progressively working your way through the precondition list. If you fail at any point, you may need to backtrack because there might have been another unification of that predicate that would succeed. Similarly, as already mentioned, there may be more than one.

So...by using unification and a properly defined <code>successors</code> function, you should be able to apply graph based search to the problem and return a "path" through the states from the initial state to the goal. You'll definitely want to use graph-based search since <code>( drive Me Store), (drive Me Home), (drive Me Store), (drive Me Home), (drive Me Store), (buy Me Store Drill), (drive Me Home)</code> is a valid plan.

Your function should return the plan...a list of actions, fully instantiated, for the agent to do in order: [a1, a2, a3]. If you pass an extra intermediate=True parameter, it should also return the resulting state of each action: [s0, a1, s1, a2, s2, a3, s3].

-----

(you can just overwrite that one and add as many others as you need). Remember to follow the **Guidelines**.


-----

So you need to implement `forward_planner` as described above. `start_state`, `goal` and `actions` should all have the layout above and be s-expressions.

Your implementation should return the plan as a **List of instantiated actions**. If `debug=True`, you should print out the intermediate states of the plan as well.

You will be solving the problem from above. Here is the start state:

In [87]:
start_state = [
    "(item Saw)",
    "(item Drill)",
    "(place Home)",
    "(place Store)",
    "(place Bank)",
    "(agent Me)",
    "(at Me Home)",
    "(at Saw Store)",
    "(at Drill Store)"
]

The goal state:

In [88]:
goal = [
    "(item Saw)",    
    "(item Drill)",
    "(place Home)",
    "(place Store)",
    "(place Bank)",    
    "(agent Me)",
    "(at Me Home)",
    "(at Drill Me)",
    "(at Saw Store)"    
]

and the actions/operators:

In [89]:
actions = {
    "drive": {
        "action": "(drive ?agent ?from ?to)",
        "conditions": [
            "(agent ?agent)",
            "(place ?from)",
            "(place ?to)",
            "(at ?agent ?from)"
        ],
        "add": [
            "(at ?agent ?to)"
        ],
        "delete": [
            "(at ?agent ?from)"
        ]
    },
    "buy": {
        "action": "(buy ?purchaser ?seller ?item)",
        "conditions": [
            "(item ?item)",
            "(place ?seller)",
            "(agent ?purchaser)",
            "(at ?item ?seller)",
            "(at ?purchaser ?seller)"
        ],
        "add": [
            "(at ?item ?purchaser)"
        ],
        "delete": [
            "(at ?item ?seller)"
        ]
    }
}

**Note** The facts for each state are really an ordered set. When comparing two states, you may need to convert them to a Set first.

<a id="apply_result"></a>
## apply_result

`apply_result` is a helper function (from Module 7) that recursively makes in-place variable substitutions in logic expressions. **Used by**: [action_condition_search](#action_condition_search), [convert_to_fact_or_action](#convert_to_fact_or_action)

* **result** dict: the resultant variable to constant mapping from recursive unification.
* **exp** list: the logic expression. Modified in-place.

**returns**: does not return anything and modifies in-place.

In [90]:
def apply_result(result: dict, exp: list) -> None:
    for i, e in enumerate(exp):
        if isinstance(e, list):
            apply_result(result, e)
        elif e in result:
            exp[i] = result[e]

In [91]:
result = {}
exp = ["A"]
apply_result(result, exp) 
assert exp == ["A"]

result = {"?x": "A"}
exp = ["?x", "B"]
apply_result(result, exp)
assert exp == ["A", "B"]

exp = ["?x", "?x"]
apply_result(result, exp)
assert exp == ["A", "A"]

result = {"?x": "A", "?y": "B"}
exp = ["?x", ["?y", "C"]]
apply_result(result, exp)
assert exp == ["A", ["B", "C"]]

<a id="repeat_locations"></a>
## repeat_locations

`repeat_locations` is a helper function used to guard against and prune the "?from" and "?to" locations being identical in the recursive DFS search for all possible unifications for an action's precondition. **Used by**: [action_condition_search](#action_condition_search)

* **frame** dict: the mapping of variable bindings.

**returns** bool: return True if there's an instance of "?to" and "?from" being the same location, otherwise returns False. 

In [92]:
def repeat_locations(frame: dict) -> bool:
    if "?from" in frame and "?to" in frame:
        if frame["?from"] == frame["?to"]:
            return True
    return False

In [93]:
frame = {"?agent": "me"}
assert repeat_locations(frame) == False
frame = {"?from": "north_pole", "?to": "south_pole"}
assert repeat_locations(frame) == False
frame = {"?agent": "me", "?from": "north_pole", "?to": "north_pole"}
assert repeat_locations(frame) == True

<a id="action_condition_search"></a>
## action_condition_search

`action_condition_search` does a recursive DFS and finds all the ways the facts can unify with the pre-conditions for an action, ensuring that ?to and ?from aren't counted as possible permutations of unifications. **Uses**: [action_condition_search](#action_condition_search), [apply_result](#apply_result), [repeat_locations](#repeat_locations), parse, unification **Used by**: [action_successors](#action_successors)

* **current_facts** list[str]: the facts associated with the current state.
* **conditions** list[str]: the conditions to unify with.
* **frame** list[str]: the compiled unifications from the search.

**returns** list[dict]: the list of unification dicts.

In [94]:
def action_condition_search(current_facts: list[str], conditions: list[str], frame: dict | None = None) -> list[dict]:
    if frame is None:
        frame = {}
    if not conditions:
        return [frame]
    next_condition = conditions[0]
    rest_conditions = conditions[1:]
    parsed_condition = parse(next_condition)
    apply_result(frame, parsed_condition)
    results = []
    for fact in current_facts:
        parsed_fact = parse(fact)
        new_frame = unification(parsed_condition, parsed_fact, frame.copy())
        if new_frame is not False and not repeat_locations(new_frame):
            results.extend(action_condition_search(current_facts, rest_conditions, new_frame))
    return results


In [95]:
conditions = ["(agent ?agent)"]
current_facts = ["(agent santa)"]
assert action_condition_search(current_facts=current_facts, conditions=conditions) == [{'?agent': 'santa'}]
conditions = ["(agent ?agent)", "(place ?from)"]
current_facts = ["(agent santa)", "(place north_pole)"]
assert action_condition_search(current_facts=current_facts, conditions=conditions) == [{'?agent': 'santa', '?from': 'north_pole'}]
conditions = ["(agent ?agent)", "(place ?from)", "(place ?to)"]
current_facts = ["(agent santa)", "(place north_pole)", "(place south_pole)"]
assert action_condition_search(current_facts=current_facts, conditions=conditions) == [{'?agent': 'santa', '?from': 'north_pole', '?to': 'south_pole'}, {'?agent': 'santa', '?from': 'south_pole', '?to': 'north_pole'}]


<a id="parenthify"></a>
## parenthify

`parenthify` is a helper function used in the process of updating the state when generating successor actions as part of planning. It takes a list of strings and converts it to our parenthetical fact/action format. **Used by**: [convert_to_fact_or_action](#convert_to_fact_or_action)

* **expr**  list[str]: the list of strings to convert.

**returns** str: return a string of fact components enclosed by parenthesis. 

In [96]:
def parenthify(expr: list[str]) -> str:
    return "(" + " ".join(expr) + ")"

In [97]:
assert parenthify(["agent", "me"]) == "(agent me)"
assert parenthify(["at", "me", "los_angeles"]) == "(at me los_angeles)"
assert parenthify(["from", "los_angeles", "to", "new_york"]) == "(from los_angeles to new_york)"

<a id="convert_to_fact_or_action"></a>
## convert_to_fact_or_action

`convert_to_fact_or_action` is a helper function used in the process of updating the state when generating successor actions as part of planning. It uses a permutation of unification bindings and a fact from the add or delete list for a particular action to convert the permutation into a state fact that can either be added or deleted from the overall state. **Uses**: [parenthify](#parenthify), [apply_result](#apply_result) **Used by**: [update_state](#update_state)

* **perm** dict: the permutation of unifications from the recursive search.
* **fact** str: the fact from the add/delete list for a particular action. 

**returns** str: return a string enclosed by parenthesis, representing the fact to be added or deleted from the state.

In [98]:
def convert_to_fact_or_action(perm: dict, fact_or_action: str) -> str:
    parsed_fact_or_action = parse(fact_or_action)
    apply_result(perm, parsed_fact_or_action)
    fact_or_action = parenthify(parsed_fact_or_action)
    return fact_or_action

In [99]:
fact = "(?agent)"
perm = {"?agent": "me"}
assert convert_to_fact_or_action(perm, fact) == "(me)"
fact = "(at ?agent ?to)"
perm = {"?agent": "me", "?to": "los_angeles"}
assert convert_to_fact_or_action(perm, fact) == "(at me los_angeles)"
fact = "(at ?item ?place)"
perm = {"?item": "tool", "?place": "shed"}
assert convert_to_fact_or_action(perm, fact) == "(at tool shed)"


<a id="update_state"></a>
## update_state

`update_state` uses a permutation of unification bindings and the add/delete lists from an action to update the facts of a successor state. **Uses**: [convert_to_fact_or_action](#convert_to_fact_or_action), **Used by**: [action_successors](#action_successors)

* **perm** dict: the permutation of unifications from the recursive search.
* **successor_state** list[str]: the updated state, based on the add/delete lists for the application action.
* **action** dict: the action to use when updating the state.

**returns** list[str]: returns the updated facts for the successor state.

In [100]:
def update_state(perm: dict, successor_state: list[str], action: dict) -> list[str]:
    for del_fact in action["delete"]:
        factified = convert_to_fact_or_action(perm, del_fact)
        if factified in successor_state:
            successor_state.remove(factified)
    for add_fact in action["add"]:
        factified = convert_to_fact_or_action(perm, add_fact)
        if factified not in successor_state:
            successor_state.append(factified)
    return successor_state

In [101]:
test_action = {"action": "(drive ?agent ?from ?to)","conditions": ["(agent ?agent)","(place ?from)","(place ?to)"],"add": ["(at ?agent ?to)"],"delete": ["(at ?agent ?from)"]}
perm = {"?agent": "me", "?from": "los_angeles", "?to": "new_york"}
successor_state = ["(agent me)","(at me los_angeles)"]
assert update_state(perm, successor_state, test_action) == ['(agent me)', '(at me new_york)']
successor_state = ["(agent me)","(at me new_york)"]
assert update_state(perm, successor_state, test_action) == ['(agent me)', '(at me new_york)']
perm = {"?agent": "me", "?from": "los_angeles", "?to": "new_york"}
successor_state = []
assert update_state(perm, successor_state, test_action) == ['(at me new_york)']

<a id="action_successors"></a>
## action_successors

`action_successors` finds the permissible actions given the current state facts and creates a successor for each permutation of state fact substitutions that make each action permissible. **Uses**: [action_condition_search](#action_condition_search), [update_state](#update_state), **Used by**: [forward_planner](#forward_planner)

* **current_facts** list[str]: the permutation of unifications from the recursive search.
* **action** dict: the action to use when updating the state.

**returns** list[tuple[list[str], str]]: returns the updated facts for the successor state as well as the action in a tuple.

In [102]:
def action_successors(current_facts: list[str], actions: dict) -> list[tuple[list[str], str]]:
    successors = []
    explored = set()
    for action in actions.values():
        permissible_action_permutations = action_condition_search(current_facts, action["conditions"])
        for perm in permissible_action_permutations:
            successor_state = current_facts.copy()
            successor = sorted(update_state(perm, successor_state, action))
            converted_action = convert_to_fact_or_action(perm, action["action"])
            if tuple(successor) not in explored:
                explored.add(tuple(successor))
                successors.append((successor, converted_action))
    return successors

In [103]:
current_facts = ["(agent me)", "(place north_pole)", "(place south_pole)", "(at me north_pole)"]
action = {"go": {"action": "(move ?agent ?from ?to)","conditions": ["(agent ?agent)","(place ?from)","(place ?to)","(at ?agent ?from)"],"add": ["(at ?agent ?to)"],"delete": ["(at ?agent ?from)"]}}
result = action_successors(current_facts, action)
assert len(result) == 1
assert (['(agent me)', '(at me south_pole)', '(place north_pole)', '(place south_pole)'], '(move me north_pole south_pole)') in result
current_facts = ["(agent me)", "(place north_pole)", "(place south_pole)", "(place equator)", "(at me north_pole)"]
result = action_successors(current_facts, action)
assert len(result) == 2
assert (['(agent me)', '(at me south_pole)', '(place equator)', '(place north_pole)', '(place south_pole)'], '(move me north_pole south_pole)') in result
assert  (['(agent me)', '(at me equator)', '(place equator)', '(place north_pole)', '(place south_pole)'], '(move me north_pole equator)') in result

<a id="check_for_goal"></a>
## check_for_goal

`check_for_goal` check to see if the current state matches the goal. **Used by**: [forward_planner](#forward_planner)

* **current_state** list[str]: the permutation of unifications from the recursive search.
* **goal** list[str]: the list of facts that comprise the goal state.

**returns** bool: returns True if the goal matches the current state, otherwise returns False.

In [104]:
def check_for_goal(current_facts: list[str], goal: list[str]) -> bool:
    state_set = set(current_facts)
    goal_set = set(goal)
    return goal_set.issubset(state_set)

In [105]:
current_facts = ["(agent smith)", "(at matrix)"]
test_goal = ["(agent smith)", "(at matrix)"]
assert check_for_goal(current_facts, test_goal)
test_goal = ["(agent neo)", "(at matrix)"]
assert not check_for_goal(current_facts, test_goal)
current_state = []
assert not check_for_goal(current_facts, test_goal)

<a id="interleave_states_and_actions"></a>
## interleave_states_and_actions

`interleave_states_and_actions` is a helper function that interleaves the ordered actions, and the resulting state from each action, for use in the forward planner debug mode, to see how the state unfolds over time. **Used by**: [forward_planner](#forward_planner)

* **states** list[list[str]]: the list of states in the path toward the matched goal.
* **actions** list[str]: the list of actions taken, in order.

**returns** list: the interleaved states and actions, i.e.  [s0, a1, s1, a2, s2, a3, s3, ...].

In [106]:
def interleave_states_and_actions(states: list[list[str]], actions: list[str]) -> list:
    result = []
    for i, state in enumerate(states):
        result.append(state)
        if i < len(actions):
            result.append(actions[i])
    return result

In [107]:
test_states = [["(agent me)", "(at north_pole)"]]
test_actions = ["(move me north_pole)"]
assert interleave_states_and_actions(test_states, test_actions) == [['(agent me)', '(at north_pole)'], '(move me north_pole)']
test_states = [["(agent me)", "(at me north_pole)"], ["(agent me)", "(at me south_pole)"]]
test_actions = ["(move me south_pole)"]
assert interleave_states_and_actions(test_states, test_actions) == [['(agent me)', '(at me north_pole)'], '(move me south_pole)', ['(agent me)', '(at me south_pole)']]
test_states = [["(agent me)", "(at me north_pole)"], ["(agent me)", "(at me south_pole)"], ["(agent me)", "(at me north_pole)"]]
test_actions = ["(move me south_pole)", "(move me north_pole)"]
assert interleave_states_and_actions(test_states, test_actions) == [['(agent me)', '(at me north_pole)'], '(move me south_pole)', ['(agent me)', '(at me south_pole)'], '(move me north_pole)', ['(agent me)', '(at me north_pole)']]

<a id="forward_planner"></a>
## forward_planner

`forward_planner`implements forward planning using the DFS psuedocode from Module 2. There's an outer search that checks if the goal has been met and generates child states based on actions, and an inner search in action_successors, that does a search on all the ways the state facts can unify with action conditions. If the goal is met, the ordered actions are returned if debug is False, and the interleaved actions and resulting states are returned if debug is True. **Uses**: [check_for_goal](#check_for_goal), [action_sucessors](#action_successors), [interleave_states_and_actions](#interleave_states_and_actions)

* **start_state** list[str]: the initial state facts for the planner.
* **goal** list[str]: the list of facts that constitute the goal state.
* **actions** dict: the action schema of possible actions, preconditions, and add/delete lists.
* **debug** bool: will add the state after each action to the returned list if True.

**returns** list: the list of actions and state after each action if debug=True, or just the list of ordered actions of debug=False.

In [108]:
def forward_planner(start_state: list[str], goal: list[str], actions: dict, debug: bool = False) -> list:
    start_facts = sorted(start_state)
    explored = set()
    frontier = [(start_facts, [], [start_facts])]
    while frontier:
        facts, plan, accumulated_facts = frontier.pop()
        if check_for_goal(facts, goal):
            return interleave_states_and_actions(accumulated_facts, plan) if debug else plan
        children = action_successors(facts, actions)
        for child_facts, action in children:
            child_facts = sorted(child_facts)
            already_in_frontier = any(tuple(frontier_facts) == tuple(child_facts) for frontier_facts, _, _ in frontier)
            if tuple(child_facts) not in explored and not already_in_frontier:
                frontier.append((child_facts, plan + [action], accumulated_facts + [child_facts]))
        explored.add(tuple(facts))
    return []

In [109]:
forward_planner(start_state=start_state, goal=goal, actions=actions, debug=False)

['(drive Me Home Store)', '(buy Me Store Drill)', '(drive Me Store Home)']

In [110]:
forward_planner(start_state=start_state, goal=goal, actions=actions, debug=True)

[['(agent Me)',
  '(at Drill Store)',
  '(at Me Home)',
  '(at Saw Store)',
  '(item Drill)',
  '(item Saw)',
  '(place Bank)',
  '(place Home)',
  '(place Store)'],
 '(drive Me Home Store)',
 ['(agent Me)',
  '(at Drill Store)',
  '(at Me Store)',
  '(at Saw Store)',
  '(item Drill)',
  '(item Saw)',
  '(place Bank)',
  '(place Home)',
  '(place Store)'],
 '(buy Me Store Drill)',
 ['(agent Me)',
  '(at Drill Me)',
  '(at Me Store)',
  '(at Saw Store)',
  '(item Drill)',
  '(item Saw)',
  '(place Bank)',
  '(place Home)',
  '(place Store)'],
 '(drive Me Store Home)',
 ['(agent Me)',
  '(at Drill Me)',
  '(at Me Home)',
  '(at Saw Store)',
  '(item Drill)',
  '(item Saw)',
  '(place Bank)',
  '(place Home)',
  '(place Store)']]

## Before You Submit...

1. Did you provide output exactly as requested?
2. Did you re-execute the entire notebook? ("Restart Kernel and Rull All Cells...")
3. If you did not complete the assignment or had difficulty please explain what gave you the most difficulty in the Markdown cell below.
4. Did you change the name of the file to `jhed_id.ipynb`?

Do not submit any other files.